<a href="https://colab.research.google.com/github/aisyashlf/Home-Credit-/blob/main/%5BAgregasi%5D_Home_Credit_Scorecard_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
enc_app_train = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_app_train.csv')
enc_app_test = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_app_test.csv')
enc_pre_app = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_pre_app.csv')
enc_cre = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_cre.csv')
enc_bur = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_bur.csv')
enc_bur_bal = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_bur_bal.csv')
enc_posh = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_pos.csv')
enc_inst = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_inst.csv')
df_sample = pd.read_csv('/content/drive/MyDrive/Home Credit/Cleaning/clean_sample.csv')


#AGREGASI

In [ ]:
import pandas as pd

def agg_numeric(df, group_var, prefix):
    """
    Agregasi semua kolom numerik di df per group_var
    dengan statistik: mean, max, min, sum, var.

    df: dataframe sumber
    group_var: nama kolom key (misal: 'SK_ID_CURR')
    prefix: prefix nama fitur (misal: 'bur', 'prev', dll.)
    """
    # pilih hanya kolom numerik
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

    # jangan agregasi key-nya
    numeric_cols = [c for c in numeric_cols if c != group_var]

    # kalau tidak ada kolom numerik selain key, langsung keluar
    if len(numeric_cols) == 0:
        return pd.DataFrame()

    agg = df.groupby(group_var)[numeric_cols].agg(['mean', 'max', 'min', 'sum', 'var'])

    # flatten multiindex kolom: (col, stat) -> prefix_col_stat
    agg.columns = [f"{prefix}_{col}_{stat}".upper()
                   for col, stat in agg.columns]

    agg = agg.reset_index()

    return agg


In [ ]:
prev_agg = agg_numeric(enc_pre_app, group_var="SK_ID_CURR", prefix="PREV")

In [ ]:
bur_agg = agg_numeric(enc_bur, group_var="SK_ID_CURR", prefix="BUR")


In [ ]:
# 1) agregasi bureau_balance per SK_ID_BUREAU
bur_bal_agg_bureau = agg_numeric(enc_bur_bal, group_var="SK_ID_BUREAU", prefix="BUR_BAL")

# 2) join ke bureau untuk bawa SK_ID_CURR
bur_bal_merged = enc_bur[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(
    bur_bal_agg_bureau,
    on="SK_ID_BUREAU",
    how="left"
)

# 3) agregasi ke level SK_ID_CURR
bur_bal_agg_curr = agg_numeric(bur_bal_merged.drop(columns=["SK_ID_BUREAU"]),
                               group_var="SK_ID_CURR",
                               prefix="BUR_BAL")


In [ ]:
enc_cre.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,SK_DPD,SK_DPD_DEF,NAME_CONTRACT_STATUS_count
0,2562384,378907,-6,56.970,135000,0.0,0.0,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,0,0,3698436
1,2582071,363914,-1,63975.555,45000,0.0,0.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,0,0,3698436
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,0,0,3698436
3,1389973,337855,-4,222767.325,225000,0.0,0.0,0.0,0.0,11795.760,...,233048.970,222286.275,1.0,1,0.0,0.0,10.0,0,0,3698436
4,1891521,126868,-1,222767.325,450000,0.0,0.0,0.0,11547.0,22924.890,...,453919.455,222286.275,0.0,1,0.0,1.0,101.0,0,0,3698436


In [ ]:
cre_agg = agg_numeric(enc_cre, group_var="SK_ID_CURR", prefix="CRE")

In [ ]:
pos_agg = agg_numeric(enc_posh, group_var="SK_ID_CURR", prefix="POS")

In [ ]:
inst_agg = agg_numeric(enc_inst, group_var="SK_ID_CURR", prefix="INST")

In [ ]:
agg_dfs = [prev_agg, bur_agg, bur_bal_agg_curr, cre_agg, pos_agg, inst_agg]

In [ ]:
train = enc_app_train.copy()

for agg_df in agg_dfs:
    if agg_df is None or agg_df.empty:
        continue
    train = train.merge(agg_df, on="SK_ID_CURR", how="left")

In [ ]:
train.to_csv("train.csv", index=False)

In [ ]:
test = enc_app_test.copy()

for agg_df in agg_dfs:
    if agg_df is None or agg_df.empty:
        continue
    test = test.merge(agg_df, on="SK_ID_CURR", how="left")

In [ ]:
test.to_csv("test.csv", index=False)